# 03 — Agent Walkthrough

> **Requires a free API key.** Set `GROQ_API_KEY` or `GOOGLE_API_KEY` before running.

## Overview

The career-gap agent is implemented as a **LangGraph** state machine with three nodes:

| Node | Role |
|------|------|
| `planner` | Reads the system prompt and decides which tool to call next |
| `tools` | Runs `search_jobs` or `analyse_postings` against live/snapshot data |
| `reflection` | Checks that all skills in the gap report are valid ESCO labels; re-prompts if not |

The graph terminates when the report is grounded (all skills are valid ESCO labels) or the
iteration cap is reached.

This notebook runs the agent end-to-end on one example CV + target role, prints the
full message/tool trace, and summarises the final gap report.

In [ ]:
# fmt: off
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside the project tree.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')
# fmt: on


## 1. Build dependencies (snapshot mode — no Adzuna key needed)

In [ ]:
from src.data.esco_loader import EscoIndex
from src.skills.esco_matcher import EscoMatcher
from src.generation.llm_client import build_chat_model, simple_complete
from src.skills.extractor import extract_skill_phrases
from src.agent.tools import AgentDeps

index = EscoIndex.load()
matcher = EscoMatcher(index)
chat_model = build_chat_model(temperature=0.0)

deps = AgentDeps(
    app_id=None,   # snapshot mode: reads data/fixtures/adzuna_snapshot.json
    app_key=None,
    matcher=matcher,
    llm=lambda p: simple_complete(p, chat_model),
)
print('Dependencies ready.')

## 2. Build the graph

In [ ]:
from src.agent.graph import build_graph

planner_model = build_chat_model(temperature=0.0)
graph = build_graph(planner_model, deps)
print('Graph compiled.')

## 3. Define an example input

In [ ]:
example_cv = """
John Smith — Data Analyst
Skills: SQL, Excel, Tableau, basic Python (pandas, matplotlib)
Experience: 3 years retail analytics, A/B test reporting, dashboard maintenance.
"""

initial_state = {
    'messages': [],
    'cv_text': example_cv,
    'role': 'Data Analyst',
    'location': 'London',
}
print('Input defined.')

## 4. Run the agent and print the trace

In [ ]:
final = graph.invoke(initial_state)

print('=== Message / tool trace ===')
for i, msg in enumerate(final['messages']):
    role = getattr(msg, 'type', type(msg).__name__)
    content_preview = str(getattr(msg, 'content', ''))[:200].replace('\n', ' ')
    tool_calls = getattr(msg, 'tool_calls', [])
    tc_str = ', '.join(tc['name'] for tc in tool_calls) if tool_calls else ''
    print(f'[{i}] {role:12s}  tool_calls=[{tc_str}]  {content_preview}')

print()
report = final.get('report')
if report:
    print(f'=== Final GapReport ===')
    print(f'Role: {report.role}  Location: {report.location}')
    print(f'Postings analysed: {report.n_postings}')
    print(f'Skill gaps ({len(report.gaps)}):')
    for gap in report.gaps:
        print(f'  {gap.skill:40s}  demand_fraction={gap.demand_fraction:.2f}')
else:
    print('No report produced (check agent trace above).')